<a href="https://colab.research.google.com/github/Ahmed-Rafi07/-MongoDB-connection-string-in-the-.env-file-/blob/main/DataMining_GA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
frames = []
for spend, visits, basket in [(15000, 40, 400),
                              (65000, 8, 8000),
                              (35000, 22, 1600)]:
    frames.append(pd.DataFrame({
        'annual_spend': rng.normal(spend, 2500, 100).clip(1000),
        'purchase_frequency': rng.normal(visits, 2, 100).clip(1),
        'avg_basket_value': rng.normal(basket, basket * 0.10, 100).clip(100)
    }))
customer_df = pd.concat(frames, ignore_index=True).round(2)
customer_df.insert(0, 'customer_id', np.arange(1, 301))
customer_df['tenure_months'] = rng.integers(1, 61, 300)
customer_df['num_complaints'] = rng.integers(0, 5, 300)
customer_df['contract_M2M'] = rng.integers(0, 2, 300)
customer_df['churned'] = (
    (customer_df['num_complaints'] >= 3) &
    (customer_df['contract_M2M'] == 1)
).astype(int)
flip = rng.choice(300, 15, replace=False)
customer_df.loc[flip, 'churned'] = 1 - customer_df.loc[flip, 'churned']

print('Dataset shape:', customer_df.shape)
print(customer_df.head().to_string(index=False))
print('Churn class counts:')
print(customer_df['churned'].value_counts().sort_index())


Dataset shape: (300, 8)
 customer_id  annual_spend  purchase_frequency  avg_basket_value  tenure_months  num_complaints  contract_M2M  churned
           1      15761.79               39.24            413.50             51               4             0        0
           2      12400.04               42.60            456.30             25               4             0        0
           3      16876.13               39.29            403.62              7               1             0        0
           4      17351.41               41.48            425.76             46               4             1        1
           5      10122.41               38.13            317.99              2               4             0        0
Churn class counts:
churned
0    232
1     68
Name: count, dtype: int64


### Task 1 - Customer Snapshot Model

**Grain:** One row represents one customer at one observation (snapshot) date.

| Proposed table | Columns | Primary key | Foreign key relationship |
|---|---|---|---|
| FactCustomerSnapshot | customer_id, snapshot_date, contract_type_key, annual_spend, purchase_frequency | customer_id+ snapshot_date | tenure_months | num_complaints
| DimContractType | contract_type_key, contract_type_name | contract_type_key | Not applicable |

**Contract mapping:**
- contract_M2M = 1 maps to Month-to-Month.
- contract_M2M = 0 maps to O maps to Fixed-Term.

**Outcome handling:** Keep the later churn label separate from the
pre-outcome snapshot. Link it using customer_id and snapshot_date.

**Measure explanation:**
- Across distinct customers at one snapshot date, annual_spend can be summed because each customer contributes one record.
- Across repeated snapshots, summing annual_spend is misleading because overlapping 12-month.
- Summing avg_basket_value does not give total sales because it represents an average purchase value, not total transaction revenue.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, recall_score

# TODO: Use only the three classification features specified in Task 2.
classification_features = ['tenure_months', 'num_complaints', 'contract_M2M']
X = customer_df[classification_features]
y = customer_df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

classifier = DecisionTreeClassifier(max_depth=4, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# TODO: Evaluate held-out labels against held-out predictions.
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=['Retained', 'Churned'],
    zero_division=0
))
churn_recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
print('Churn recall:', round(churn_recall, 3))


Training rows: 240 Test rows: 60
              precision    recall  f1-score   support

    Retained       0.94      0.98      0.96        46
     Churned       0.92      0.79      0.85        14

    accuracy                           0.93        60
   macro avg       0.93      0.88      0.90        60
weighted avg       0.93      0.93      0.93        60

Churn recall: 0.786


**Observed churn recall:** 0.786.
This means the model caught 78.6% of the actual churners in the test set.
A missed churner means the business loses an opportunity to retain a customer; an unnecessary offer means a retention offer is given to a customer who would have stayed anyway.
Given the business priority and offer cost, recall matters because identifying more actual churners helps the business take preventive action.
Overall accuracy alone may be misleading because it does not show how many actual churners were correctly identified.


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

# Select only the three behavioural features specified in Task 3.
cluster_features = ['annual_spend', 'purchase_frequency', 'avg_basket_value']
X_cluster = customer_df[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

score_rows = []
for k in [2, 3, 4]:
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10)
    candidate_labels = candidate.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, candidate_labels)
    score_rows.append({'K': k, 'silhouette_score': score})

scores = pd.DataFrame(score_rows)
print('Silhouette comparison:')
print(scores.to_string(index=False))

# Based on the best silhouette score, we will inspect the output and finalize K.
# Since we want to define the final model, we can see the silhouette scores first. Let's programmatically choose the best K.
best_k = int(scores.loc[scores['silhouette_score'].idxmax(), 'K'])
print(f'Programmatically chosen best K: {best_k}')

chosen_k = best_k
kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Preserve customer_df; add labels only to a separate results copy.
segmented_customers = customer_df.copy()
segmented_customers['cluster'] = cluster_labels
print('\nCluster sizes:')
print(segmented_customers['cluster'].value_counts().sort_index())

# Compute means from original-unit columns, NOT X_scaled.
profile = segmented_customers.groupby('cluster')[cluster_features].mean().round(2)
print('\nCluster profiles in original units:')
print(profile.to_string())


Silhouette comparison:
 K  silhouette_score
 2          0.757533
 3          0.855209
 4          0.683024
Programmatically chosen best K: 3

Cluster sizes:
cluster
0    100
1    100
2    100
Name: count, dtype: int64

Cluster profiles in original units:
         annual_spend  purchase_frequency  avg_basket_value
cluster                                                    
0            65256.23                7.91           7926.91
1            34720.58               22.08           1573.72
2            14874.33               39.98            397.51


**Why scale?** Distance-based algorithms like K-Means are sensitive to the scale of the features; without scaling, features with larger numeric ranges (such as `annual_spend`) would dominate the distance metrics over features with smaller ranges (such as `purchase_frequency`).

**Chosen K:** 3. The scores support this choice because K=3 has the highest silhouette score (0.855), indicating well-defined, distinct clusters with high cohesion and low overlap compared to K=2 (0.758) or K=4 (0.683).

**Segment names and evidence:**
- **Cluster 0:** High-Value, Low-Frequency VIPs; mean annual spend = 65,256.23 INR, mean purchase frequency = 7.91 times/month, and mean basket value = 7,926.91 INR.
- **Cluster 1:** Mid-Value, Moderate-Frequency Customers; mean annual spend = 34,720.58 INR, mean purchase frequency = 22.08 times/month, and mean basket value = 1,573.72 INR.
- **Cluster 2:** Low-Value, High-Frequency Budget Shoppers; mean annual spend = 14,874.33 INR, mean purchase frequency = 39.98 times/month, and mean basket value = 397.51 INR.

Cluster IDs are identifiers rather than rankings because they are categorical assignments by K-Means and do not inherently imply order or value from low to high across all metric dimensions.

### Task 4 - Recommend a Retention Approach

Classification more directly answers “Who is likely to churn?” because it is a supervised model trained directly on historic ground-truth labels (`churned`).
The model's test-set churn recall was 0.786, so a practical retention use is to automatically flag high-risk customers with several complaints on Month-to-Month contracts for targeted proactive outreach.
Clustering adds deep behavioural context (spending patterns, average basket sizes, and frequencies) that individual churn predictions do not directly describe.
The High-Value, Low-Frequency VIPs segment (Cluster 0) has a mean annual spend of 65,256.23 INR and a mean purchase frequency of 7.91 times/month, suggesting they make fewer, highly premium purchases.
Use these findings as strategic guideposts to tailor retention packages and marketing strategies, not proof that a feature or segment causes churn.